# LangChain: Models, Prompts and Output Parsers (Anthropic / Claude)

## Outline

- Direct API calls to **Anthropic**
- API calls through LangChain:
  - Prompts
  - Models (`ChatAnthropic`)
  - Output parsers
- Get your **Anthropic API key**

> **Note:** LLMs do not always produce the same results. When you run this notebook, you may get slightly different answers than those in the video.

> **Packages:** `pip install anthropic langchain langchain-anthropic python-dotenv` (and `langchain-core` if your LangChain version expects it).

> **If you see `NotFoundError` / 404 for `model:`:** the model ID is wrong or retired. Update `llm_model` in the next code cell to a current ID from [Anthropic’s models page](https://docs.anthropic.com/en/docs/about-claude/models).


## Setup: environment and API key

Create a `.env` file in this folder (or your project root) with:

```
ANTHROPIC_API_KEY=sk-ant-...
```

In [1]:
# !pip install python-dotenv
# !pip install anthropic
# !pip install langchain langchain-anthropic

import os

import anthropic
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())  # read local .env file

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

In [3]:
# Default: Claude Haiku 4.5 snapshot. Older IDs like claude-3-5-haiku-20241022 return API 404 (removed).
# Docs: https://docs.anthropic.com/en/docs/about-claude/models — try claude-sonnet-4-6 for higher quality.
llm_model = "claude-haiku-4-5-20251001"


## Messages API: Anthropic (direct)

Start with direct calls to the Anthropic Messages API.

- **`system`** (optional): stable instructions for *how* the model should behave (tone, format, safety). Not the same as a user turn.
- **`messages`**: the conversation. Each item has a **`role`** (`"user"` or `"assistant"`) and **`content`**. The role name is fixed by the API; the **content changes** every time. For multi-turn chats you append more `{role, content}` objects (user → assistant → user → …).

In [5]:
# Optional: instructions for the model (behavior, style). Separate from the user's question.
DEFAULT_SYSTEM = (
    "You are a helpful assistant for someone learning the Anthropic Messages API. "
    "Give clear, accurate answers. When the user asks for code, prefer short, runnable examples."
)


def get_completion(prompt, model=llm_model, system=DEFAULT_SYSTEM):
    message = client.messages.create(
        model=model,
        max_tokens=1024,
        temperature=0.1,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    # Join all text blocks (some models return thinking + text)
    parts = []
    for block in message.content:
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
    if not parts:
        raise ValueError(f"No text blocks in response: {message.content!r}")
    return "".join(parts)


get_completion("What is 1+1?")


"1 + 1 = 2\n\nThat's basic arithmetic! Is there anything about the Anthropic Messages API you'd like to learn about?"

In [6]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

style = """American English \
in a calm and respectful tone
"""

prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)
response = get_completion(prompt)
response

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



"# Translated Text\n\nI'm quite frustrated that my blender lid came off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty doesn't cover the cost of cleaning up my kitchen. I would really appreciate your help with this matter."

## Messages API: LangChain

Do the same using LangChain: **model**, **prompt template**, then invoke the chat model.

### Model

In [7]:
# !pip install --upgrade langchain langchain-anthropic

from langchain_anthropic import ChatAnthropic

# To control randomness and creativity, use temperature = 0.0
chat = ChatAnthropic(
    temperature=0.0,
    model=llm_model,
    max_tokens=1024,
)
chat

ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0.0, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={})

### Prompt template

In [8]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)

prompt_template.messages[0].prompt
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [9]:
prompt_template.messages[0].prompt


PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [ ]:
customer_style = """American English \
in a calm and respectful tone
"""

customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

customer_messages = prompt_template.format_messages(
    style=customer_style,
    text=customer_email,
)
print(type(customer_messages))
print(type(customer_messages[0]))
print(customer_messages[0])

customer_response = chat(customer_messages)
print(customer_response.content)

In [27]:
print(customer_messages[0])


content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [28]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply,
)

print(service_messages[0].content)
service_response = chat(service_messages)
print(service_response.content)

For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!


```json
{
  "gift": false,
  "delivery_days": -1,
  "price_value": []
}
```


## Output parsers

Target structured output shape:

```json
{
  "gift": false,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}
```

In [29]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

messages = prompt_template.format_messages(text=customer_review)
chat = ChatAnthropic(
    temperature=0.0,
    model=llm_model,
    max_tokens=1024,
)
response = chat(messages)
print(response.content)
type(response.content)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]
```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]
}
```


str

In [ ]:
# response.content is the model's text (a str), not a dict — so .get() does not exist.
print(type(response.content))  # <class 'str'>

# Uncomment the next line to see the intentional AttributeError from the lesson:
# response.content.get("gift")

# The next cells fix this with StructuredOutputParser → a real dict with .get("gift").

### Parse the LLM output string into a Python dictionary

Use `ResponseSchema` + `StructuredOutputParser` and append `format_instructions` to the prompt.

In [30]:
from langchain_classic.output_parsers import ResponseSchema
from langchain_classic.output_parsers import StructuredOutputParser

gift_schema = ResponseSchema(
    name="gift",
    description="Was the item purchased\
    as a gift for someone else? \
    Answer True if yes,\
    False if not or unknown.",
)
delivery_days_schema = ResponseSchema(
    name="delivery_days",
    description="How many days\
    did it take for the product\
    to arrive? If this \
    information is not found,\
    output -1.",
)
price_value_schema = ResponseSchema(
    name="price_value",
    description="Extract any\
    sentences about the value or \
    price, and output them as a \
    comma separated Python list.",
)

response_schemas = [
    gift_schema,
    delivery_days_schema,
    price_value_schema,
]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"gift": string  // Was the item purchased    as a gift for someone else?     Answer True if yes,    False if not or unknown.
	"delivery_days": string  // How many days    did it take for the product    to arrive? If this     information is not found,    output -1.
	"price_value": string  // Extract any    sentences about the value or     price, and output them as a     comma separated Python list.
}
```


In [31]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(
    text=customer_review,
    format_instructions=format_instructions,
)
print(messages[0].content)

response = chat(messages)
print(response.content)

output_dict = output_parser.parse(response.content)
output_dict
type(output_dict)
output_dict.get("delivery_days")

For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,and output them as a comma separated Python list.

text: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.


The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```

'2'

In [33]:
print(response.content)
type(output_dict)

```json
{
	"gift": "True",
	"delivery_days": "2",
	"price_value": "['It\\'s slightly more expensive than the other leaf blowers out there, but I think it\\'s worth it for the extra features.']"
}
```


dict

---

**Reminder:** Download or commit this notebook so your work is saved locally.